# **Semana 9: Pipelines - ETL vs ELT, Batch vs Streaming (NB1 - Conceptual)**

## **Propósito de la Sesión**
Construir pipelines de datos simples y comprender las diferencias fundamentales entre los paradigmas ETL y ELT, así como entre procesamiento batch y streaming. Estos conceptos son la base para diseñar arquitecturas de datos escalables y eficientes en proyectos de IA y analítica.

### **Objetivos de Aprendizaje**
Al finalizar este notebook, el estudiante será capaz de:
1. **Diferenciar** entre los enfoques ETL (Extract, Transform, Load) y ELT (Extract, Load, Transform).
2. **Implementar** un pipeline ETL extrayendo datos de una API, transformándolos con pandas y cargándolos a SQLite.
3. **Implementar** un pipeline ELT cargando datos crudos a SQLite y transformándolos con SQL.
4. **Distinguir** entre procesamiento batch y streaming, y simular este último con `time.sleep()`.
5. **Aplicar** una técnica de carga incremental para simular actualizaciones eficientes de datos.

## **Configuración Inicial**

Importamos las librerías necesarias y configuramos el entorno.

In [ ]:
# Instalación de librerías necesarias
!pip install pandas requests --quiet

# Importación de librerías
import pandas as pd
import requests
import sqlite3
import json
import time
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import random

# Configuración de visualización
sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)

print("Librerías importadas correctamente.")
print(f"Versión de pandas: {pd.__version__}")

---
## **Parte 1: Fundamentos - ETL vs ELT**

### **¿Qué es un pipeline de datos?**

Un **pipeline de datos** es una serie de pasos que procesan y mueven datos desde una fuente (origen) hacia un destino (como un data warehouse), aplicando transformaciones en el camino.

### **ETL (Extract, Transform, Load)**

Es el enfoque tradicional:
1. **Extract**: Extraer datos de fuentes (APIs, bases de datos, archivos).
2. **Transform**: Aplicar limpieza, validaciones, uniones, agregaciones en un servidor intermedio (staging).
3. **Load**: Cargar los datos transformados al data warehouse.

✅ **Ventajas**: Control total de la transformación, menor carga en el DW.
❌ **Desventajas**: Requiere infraestructura adicional, mayor latencia.

### **ELT (Extract, Load, Transform)**

Con la potencia de los data warehouses modernos en la nube:
1. **Extract**: Extraer datos de fuentes.
2. **Load**: Cargar los datos **en bruto** directamente al data warehouse.
3. **Transform**: Realizar las transformaciones dentro del DW (generalmente con SQL).

✅ **Ventajas**: Simplicidad, agilidad (datos crudos disponibles rápido), aprovecha la escalabilidad del DW.
❌ **Desventajas**: Costo de almacenamiento y cómputo en el DW, no ideal para datos no estructurados.

---
## **Parte 2: Implementación de ETL con API y pandas**

En este primer pipeline, haremos **ETL**:
*   **Extract**: Consumiremos datos de una API pública (JSONPlaceholder).
*   **Transform**: Limpiaremos y enriqueceremos los datos con pandas.
*   **Load**: Cargaremos los datos transformados a una base de datos SQLite.

### **2.1. Extract: Obtener datos de la API**

In [ ]:
# Configuración de la API
api_url_posts = 'https://jsonplaceholder.typicode.com/posts'
api_url_users = 'https://jsonplaceholder.typicode.com/users'

print("Extrayendo datos de la API...")

# Extraer posts
response_posts = requests.get(api_url_posts)
if response_posts.status_code == 200:
    posts_data = response_posts.json()
    print(f"  - {len(posts_data)} posts extraídos.")
else:
    print(f"Error al extraer posts: {response_posts.status_code}")
    posts_data = []

# Extraer usuarios
response_users = requests.get(api_url_users)
if response_users.status_code == 200:
    users_data = response_users.json()
    print(f"  - {len(users_data)} usuarios extraídos.")
else:
    print(f"Error al extraer usuarios: {response_users.status_code}")
    users_data = []

### **2.2. Transform: Limpiar y enriquecer con pandas**

En esta fase de transformación, vamos a:
1.  Convertir las listas de diccionarios a DataFrames.
2.  Filtrar posts relevantes (por ejemplo, los que tienen títulos con más de 30 caracteres).
3.  Enriquecer los posts con el nombre del autor, uniendo con los datos de usuarios.
4.  Crear una columna de longitud del título para análisis.

In [ ]:
print("Transformando datos con pandas...")

# Convertir a DataFrames
df_posts_raw = pd.DataFrame(posts_data)
df_users = pd.DataFrame(users_data)

print(f"  - DataFrame posts: {df_posts_raw.shape}")
print(f"  - DataFrame users: {df_users.shape}")

# 1. Filtrar posts: solo los que tienen título con más de 30 caracteres
df_posts_filtered = df_posts_raw[df_posts_raw['title'].str.len() > 30].copy()
print(f"  - Posts con título >30 caracteres: {len(df_posts_filtered)}")

# 2. Enriquecer: añadir nombre del autor
#   Para ello, renombramos la columna 'id' de users a 'userId' para la unión
df_users_renamed = df_users.rename(columns={'id': 'userId'})
df_posts_enriched = df_posts_filtered.merge(
    df_users_renamed[['userId', 'name', 'email']], 
    on='userId', 
    how='left'
)

# 3. Añadir columna de longitud del título
df_posts_enriched['title_length'] = df_posts_enriched['title'].str.len()

# 4. Seleccionar y ordenar columnas para el resultado final
df_posts_final = df_posts_enriched[[
    'id', 'userId', 'name', 'email', 'title', 'body', 'title_length'
]].rename(columns={'name': 'author_name'})

print("\nDatos transformados (primeras 5 filas):")
display(df_posts_final.head())

# Visualizar distribución de longitud de títulos
plt.figure(figsize=(8, 4))
plt.hist(df_posts_final['title_length'], bins=15, color='skyblue', edgecolor='black')
plt.title('Distribución de la Longitud de Títulos (post-transformación)')
plt.xlabel('Longitud del Título')
plt.ylabel('Frecuencia')
plt.show()

### **2.3. Load: Cargar datos transformados a SQLite**

Finalmente, cargamos los datos ya transformados en nuestra base de datos destino (SQLite).

In [ ]:
# Conectar a la base de datos SQLite (se crea el archivo si no existe)
conn = sqlite3.connect('pipeline.db')
print("Conectado a base de datos 'pipeline.db'")

# Cargar el DataFrame transformado en una tabla llamada 'posts_etl'
df_posts_final.to_sql('posts_etl', conn, if_exists='replace', index=False)
print("Datos cargados en tabla 'posts_etl'")

# Verificar la carga
df_check = pd.read_sql_query("SELECT COUNT(*) as total FROM posts_etl", conn)
print(f"Total de registros en 'posts_etl': {df_check['total'][0]}")

# Mostrar un ejemplo de los datos cargados
print("\nPrimeros 3 registros en la base de datos:")
display(pd.read_sql_query("SELECT * FROM posts_etl LIMIT 3", conn))

---
## **Parte 3: Implementación de ELT con Carga Cruda y Transformación SQL**

Ahora haremos **ELT**:
*   **Extract**: Extraeremos datos (simulando una fuente nueva, por ejemplo, comentarios).
*   **Load**: Cargaremos los datos **en crudo** a SQLite.
*   **Transform**: Realizaremos las transformaciones directamente con SQL dentro de la base de datos.

### **3.1. Extract y Load: Obtener datos crudos y cargarlos**

Extraemos comentarios de la API y los cargamos sin transformar.

In [ ]:
# Extraer comentarios de la API
api_url_comments = 'https://jsonplaceholder.typicode.com/comments'
print("Extrayendo comentarios de la API...")

response_comments = requests.get(api_url_comments)
if response_comments.status_code == 200:
    comments_data = response_comments.json()
    print(f"  - {len(comments_data)} comentarios extraídos.")
else:
    print(f"Error: {response_comments.status_code}")
    comments_data = []

# Convertir a DataFrame (datos crudos, sin transformar)
df_comments_raw = pd.DataFrame(comments_data)
print(f"  - DataFrame crudo dimensiones: {df_comments_raw.shape}")

# Cargar datos crudos a SQLite en una tabla 'comments_raw'
df_comments_raw.to_sql('comments_raw', conn, if_exists='replace', index=False)
print("Datos crudos cargados en tabla 'comments_raw'")

# Verificar
count_raw = pd.read_sql_query("SELECT COUNT(*) as total FROM comments_raw", conn)
print(f"Total registros en 'comments_raw': {count_raw['total'][0]}")

### **3.2. Transform: Procesar datos con SQL dentro de la base de datos**

Ahora aplicaremos transformaciones usando SQL, aprovechando la capacidad de procesamiento de la base de datos (en este caso SQLite, pero en la nube sería Snowflake/BigQuery/Redshift).

Transformaciones a realizar:
1.  Crear una tabla `comments_enriched` que:
    *   Filtre comentarios con email válido (que contenga '@').
    *   Extraiga el dominio del email.
    *   Calcule la longitud del comentario.
2.  Crear un resumen por dominio: cantidad de comentarios y longitud promedio.

In [ ]:
print("Aplicando transformaciones con SQL...")

# --- Transformación 1: Crear tabla enriquecida ---
query_enriched = """
CREATE TABLE comments_enriched AS
SELECT
    id,
    postId,
    name,
    email,
    body,
    LENGTH(body) AS comment_length,
    SUBSTR(email, INSTR(email, '@') + 1) AS email_domain
FROM comments_raw
WHERE email LIKE '%@%'  -- emails válidos
"""

cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS comments_enriched")
cursor.execute(query_enriched)
conn.commit()
print("Tabla 'comments_enriched' creada.")

# Verificar
df_enriched = pd.read_sql_query("SELECT * FROM comments_enriched LIMIT 5", conn)
print("\nPrimeros 5 registros en 'comments_enriched':")
display(df_enriched)

# --- Transformación 2: Resumen por dominio ---
query_summary = """
SELECT
    email_domain,
    COUNT(*) AS comment_count,
    ROUND(AVG(comment_length), 2) AS avg_comment_length
FROM comments_enriched
GROUP BY email_domain
ORDER BY comment_count DESC
LIMIT 10
"""

df_domain_summary = pd.read_sql_query(query_summary, conn)
print("\nTop 10 dominios de email por cantidad de comentarios:")
display(df_domain_summary)

# Visualizar
plt.figure(figsize=(10, 4))
plt.bar(df_domain_summary['email_domain'], df_domain_summary['comment_count'], color='coral')
plt.title('Top 10 Dominios de Email por Cantidad de Comentarios')
plt.xlabel('Dominio')
plt.ylabel('Cantidad de Comentarios')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## **Parte 4: Batch vs Streaming (simulación)**

### **Conceptos Clave**

*   **Batch (Lotes)**: Los datos se procesan en bloques a intervalos regulares (cada hora, cada día). Es eficiente para grandes volúmenes, pero tiene latencia.
*   **Streaming (Tiempo Real)**: Los datos se procesan continuamente a medida que llegan, con baja latencia (milisegundos/segundos).

### **Simulación de Streaming con `time.sleep()`**

Aunque no tenemos un sistema de streaming real, podemos simular la llegada continua de datos usando un bucle con pausas y procesar cada lote pequeño.

In [ ]:
print("--- SIMULACIÓN DE PROCESAMIENTO STREAMING ---")
print("Cada 'evento' simula la llegada de un nuevo post.")

# Simular una fuente de datos: lista de eventos (posts)
sample_posts = posts_data[:10]  # Tomamos 10 posts para simular

for i, post in enumerate(sample_posts):
    # Simular la llegada de un nuevo evento
    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Evento {i+1} recibido: Post ID {post['id']}")

    # Procesar el evento (transformación simple en tiempo real)
    title_length = len(post['title'])
    print(f"    Procesando: Título '{post['title'][:30]}...' (longitud: {title_length})")

    # Almacenar en una tabla de 'streaming' (simulado)
    # En un caso real, aquí se publicaría en Kafka o se insertaría en una base de datos de eventos
    # Simulamos inserción en SQLite
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS posts_stream (
            id INTEGER PRIMARY KEY,
            title TEXT,
            title_length INTEGER,
            received_at TIMESTAMP
        )
    """)
    cursor.execute(
        "INSERT OR IGNORE INTO posts_stream (id, title, title_length, received_at) VALUES (?, ?, ?, ?)",
        (post['id'], post['title'], title_length, datetime.now())
    )
    conn.commit()
    print(f"    Evento almacenado en 'posts_stream'.")

    # Simular el intervalo de llegada de datos (ej. cada 2 segundos)
    time.sleep(2)

print("\n--- SIMULACIÓN FINALIZADA ---")

# Mostrar los datos almacenados durante el streaming simulado
df_stream = pd.read_sql_query("SELECT * FROM posts_stream ORDER BY received_at", conn)
print("\nDatos almacenados en 'posts_stream' durante la simulación:")
display(df_stream)

---
## **Parte 5: Técnica de Carga Incremental (simulación)**

En pipelines reales, no recargamos todo el data warehouse cada día, sino que aplicamos **cargas incrementales** para actualizar solo los datos nuevos o modificados. Esto ahorra tiempo y recursos.

### **Técnica: Marca de Agua (Watermark)**

Guardamos la fecha/hora de la última carga y extraemos solo los registros posteriores a esa marca.

### **Simulación**

Simularemos que cada día llegan nuevos posts. En lugar de recargar todo, cargaremos solo los posts del día actual.

In [ ]:
print("--- SIMULACIÓN DE CARGA INCREMENTAL (WATERMARK) ---")

# Crear una tabla de control para guardar la última fecha de carga
cursor.execute("""
    CREATE TABLE IF NOT EXISTS control_cargas (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        tabla TEXT,
        ultima_fecha TIMESTAMP,
        fecha_actualizacion TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")
conn.commit()

# Función para obtener la última fecha de carga
def obtener_ultima_fecha(tabla):
    df = pd.read_sql_query(f"SELECT ultima_fecha FROM control_cargas WHERE tabla = '{tabla}' ORDER BY fecha_actualizacion DESC LIMIT 1", conn)
    if len(df) == 0:
        # Si no hay registro, usar una fecha antigua (ej. '2020-01-01')
        return '2020-01-01'
    else:
        return df['ultima_fecha'][0]

# Función para actualizar la marca de agua
def actualizar_marca_agua(tabla, nueva_fecha):
    cursor.execute(
        "INSERT INTO control_cargas (tabla, ultima_fecha) VALUES (?, ?)",
        (tabla, nueva_fecha)
    )
    conn.commit()
    print(f"  Marca de agua actualizada para '{tabla}': {nueva_fecha}")

# --- Simulación de cargas diarias ---
print("\nSimulando cargas diarias durante 3 días...\n")

for dia in range(1, 4):
    fecha_simulada = f"2025-03-0{dia}"  # Ej: 2025-03-01, 2025-03-02, ...
    print(f"\n--- DÍA {dia} ({fecha_simulada}) ---")

    # 1. Obtener la última fecha de carga para 'posts'
    ultima_fecha = obtener_ultima_fecha('posts')
    print(f"  Última carga registrada: {ultima_fecha}")

    # 2. Simular datos fuente con fecha (aquí inventamos datos)
    #    En un caso real, sería: SELECT * FROM source WHERE fecha > ultima_fecha
    datos_nuevos = []
    for i in range(random.randint(2, 5)):  # Cada día llegan entre 2 y 5 posts nuevos
        datos_nuevos.append({
            'id': dia * 100 + i,
            'title': f'Post nuevo del día {dia} (ID: {dia*100 + i})',
            'fecha': fecha_simulada
        })

    df_nuevos = pd.DataFrame(datos_nuevos)
    print(f"  Datos extraídos para {fecha_simulada}: {len(df_nuevos)} registros")

    # 3. Cargar solo los datos nuevos en la tabla destino
    df_nuevos.to_sql('posts_incremental', conn, if_exists='append', index=False)
    print(f"  {len(df_nuevos)} registros cargados en 'posts_incremental'.")

    # 4. Actualizar la marca de agua para el próximo día
    actualizar_marca_agua('posts', fecha_simulada)

print("\n--- SIMULACIÓN DE CARGA INCREMENTAL FINALIZADA ---")

# Mostrar el resultado final
df_incremental = pd.read_sql_query("SELECT * FROM posts_incremental ORDER BY fecha, id", conn)
print("\nDatos en 'posts_incremental' después de las cargas incrementales:")
display(df_incremental)

print("\nHistorial de marcas de agua:")
display(pd.read_sql_query("SELECT * FROM control_cargas", conn))

---
## **Ejercicios Prácticos para el Estudiante**

Ahora aplica lo aprendido con estos ejercicios.

### **Ejercicio 1: ETL con otra fuente de datos**

Utiliza el endpoint `/todos` de JSONPlaceholder (`https://jsonplaceholder.typicode.com/todos`) para crear un pipeline ETL que:
1.  Extraiga los datos.
2.  Transforme: filtre solo los `completed = true`, y añada una columna que indique la longitud del título.
3.  Cargue el resultado en una tabla `todos_completados_etl` en SQLite.

Muestra las primeras 5 filas de la tabla resultante.

In [ ]:
# --- INICIO DE TU CÓDIGO ---

# 1. Extract
url_todos = 'https://jsonplaceholder.typicode.com/todos'
# ...

# 2. Transform con pandas
# ...

# 3. Load a SQLite
# ...

# 4. Mostrar resultado
# ...

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 2: ELT - Transformación SQL más compleja**

Partiendo de la tabla `comments_raw` ya cargada, crea una nueva tabla `comments_summary` que agrupe por `postId` y calcule:
*   Número total de comentarios por post.
*   Longitud promedio del comentario por post.
*   Email más frecuente (moda) en los comentarios de cada post (para este punto, si es muy complejo, puedes calcular el primer email por orden alfabético como aproximación).

Muestra los primeros 5 registros de la tabla resultante.

In [ ]:
# --- INICIO DE TU CÓDIGO ---

# Escribe aquí tu consulta SQL
query_ej2 = """

"""

# Ejecutar y mostrar resultado

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 3: Simulación de streaming con agregación en ventana de tiempo**

Modifica la simulación de streaming (Parte 4) para que, en lugar de solo almacenar cada evento, mantenga un contador actualizado de cuántos eventos se han recibido en los últimos 10 segundos (una ventana deslizante simple).

*Pista: puedes mantener una lista con los timestamps de los eventos y, cada vez que llega uno nuevo, eliminar los que tengan más de 10 segundos y contar los restantes.*

In [ ]:
# --- INICIO DE TU CÓDIGO ---

# Implementa aquí tu simulación

# --- FIN DE TU CÓDIGO ---

---
## **Conclusiones de la Semana 9**

En este notebook hemos:

✅ **Diferenciado ETL vs ELT**: Comprendimos cuándo es preferible cada enfoque.

✅ **Implementado un pipeline ETL completo**: Extracción desde API, transformación con pandas y carga a SQLite.

✅ **Implementado un pipeline ELT**: Carga de datos crudos y transformación con SQL.

✅ **Simulado procesamiento batch y streaming**: Usando bucles con pausas para imitar la llegada continua de datos.

✅ **Aplicado carga incremental**: Con la técnica de watermark, simulando actualizaciones eficientes.

Estos conceptos son la base para construir pipelines robustos y eficientes en entornos de producción de datos.

---
**Fin del Notebook - Semana 9**